# IEEE-CIS (Card) — XGBoost Fraud Detection Training

**Fully self-contained** — no external imports needed. Just add the IEEE-CIS dataset and run.

**Dataset:** [IEEE-CIS Fraud Detection](https://www.kaggle.com/competitions/ieee-fraud-detection/data)  
**Output:** `artifacts/` folder downloadable from the Output tab as a zip.

## 1. Install Dependencies

In [ ]:
!pip install -q xgboost scikit-learn

## 2. Configuration

In [ ]:
# Set to None for full training, or a small number (for example 20000) for dry-run
SAMPLE_ROWS = 20000

ARTIFACT_VERSION = "v1"
N_ESTIMATORS = 300
MAX_DEPTH = 6
LEARNING_RATE = 0.1
TRAIN_RATIO = 0.8

# Kaggle dataset path candidates — first existing path will be used
TXN_PATH_CANDIDATES = [
    "/kaggle/input/ieee-fraud-detection/train_transaction.csv",
    "/kaggle/input/ieee-cis-fraud-detection/train_transaction.csv",
]
IDENTITY_PATH_CANDIDATES = [
    "/kaggle/input/ieee-fraud-detection/train_identity.csv",
    "/kaggle/input/ieee-cis-fraud-detection/train_identity.csv",
]
OUTPUT_DIR = "/kaggle/working/artifacts"

## 3. Load & Join Data

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np

TXN_PATH = next((p for p in TXN_PATH_CANDIDATES if Path(p).exists()), None)
IDENTITY_PATH = next((p for p in IDENTITY_PATH_CANDIDATES if Path(p).exists()), None)
if TXN_PATH is None or IDENTITY_PATH is None:
    raise FileNotFoundError(
        "Could not find IEEE-CIS competition files. Update path candidates for your Kaggle mount."
    )

IDENTITY_USECOLS = ["TransactionID", "DeviceType", "DeviceInfo", "id_30", "id_31", "id_33"]
read_kwargs = {"nrows": SAMPLE_ROWS} if SAMPLE_ROWS is not None else {}

print("Loading transaction data...")
txn_df = pd.read_csv(TXN_PATH, **read_kwargs)
print("Loading identity data...")
identity_df = pd.read_csv(IDENTITY_PATH, usecols=IDENTITY_USECOLS)

if SAMPLE_ROWS is not None:
    print(f"Running dry-run with first {SAMPLE_ROWS} transaction rows")
    identity_df = identity_df[identity_df["TransactionID"].isin(txn_df["TransactionID"])]

print(f"Transaction data path: {TXN_PATH}")
print(f"Identity data path: {IDENTITY_PATH}")
print(f"Transactions: {txn_df.shape}")
print(f"Identity: {identity_df.shape}")
print(f"Fraud rate: {txn_df['isFraud'].mean():.4f}")

# Left join
print("Joining transaction + identity...")
merged = txn_df.merge(identity_df, on="TransactionID", how="left")

# Build uid
UID_FIELDS = ["card1", "card2", "card3", "card5", "addr1", "addr2"]
for col in UID_FIELDS:
    if col not in merged.columns:
        merged[col] = np.nan
parts = [merged[c].where(merged[c].notna(), "NA").astype(str) for c in UID_FIELDS]
merged["uid"] = parts[0].str.cat(parts[1:], sep="_")

# identity_present — same 5 fields as Phase 4A serving
ID_FIELDS = ["DeviceType", "DeviceInfo", "id_30", "id_31", "id_33"]
present_cols = [c for c in ID_FIELDS if c in merged.columns]
merged["identity_present"] = merged[present_cols].notna().any(axis=1).astype(int) if present_cols else 0

print(f"Joined: {merged.shape}")
print(f"identity_present rate: {merged['identity_present'].mean():.4f}")

## 4. Feature Engineering

9 causal features — uid-based history only uses prior rows.

In [ ]:
import re

EPSILON = 1e-6
SECONDS_24H = 86400

IEEE_CIS_TRAINING_FEATURES = [
    "uid_prior_frequency",
    "amt_to_uid_median_ratio",
    "uid_txn_count_24h",
    "uid_amt_sum_24h",
    "uid_product_novelty",
    "email_domain_mismatch",
    "new_device_for_uid",
    "dist1_to_uid_median_ratio",
    "identity_present",
]

IEEE_CIS_ONLINE_FEATURES = [
    "uid_prior_frequency",
    "amt_to_uid_median_ratio",
    "uid_txn_count_24h",
    "email_domain_mismatch",
    "new_device_for_uid",
    "identity_present",
]

def device_signature(row):
    parts = []
    for field in ["DeviceType", "DeviceInfo", "id_30", "id_31", "id_33"]:
        value = row.get(field)
        parts.append(str(value) if pd.notna(value) else "NA")
    if all(part == "NA" for part in parts):
        return None
    return "_".join(parts)

def build_ieee_features(df):
    """Build all 9 IEEE-CIS training features causally."""
    df = df.sort_values("TransactionDT").reset_index(drop=True)

    raw_c_features = sorted(c for c in df.columns if re.fullmatch(r"C\d+", c))
    raw_d_features = sorted(c for c in df.columns if re.fullmatch(r"D\d+", c))
    raw_m_features = sorted(c for c in df.columns if re.fullmatch(r"M\d+", c))
    raw_v_features = sorted(c for c in df.columns if re.fullmatch(r"V\d+", c))

    # email_domain_mismatch (vectorized)
    has_p = df.get("P_emaildomain", pd.Series(dtype=str)).notna()
    has_r = df.get("R_emaildomain", pd.Series(dtype=str)).notna()
    both = has_p & has_r
    differ = df.get("P_emaildomain", pd.Series(dtype=str)) != df.get("R_emaildomain", pd.Series(dtype=str))
    df["email_domain_mismatch"] = (both & differ).astype(int)

    if "identity_present" not in df.columns:
        df["identity_present"] = 0

    # History-dependent features
    n = len(df)
    uid_prior_frequency = np.zeros(n, dtype=np.int64)
    amt_to_uid_median_ratio = np.zeros(n, dtype=np.float64)
    uid_txn_count_24h = np.zeros(n, dtype=np.int64)
    uid_amt_sum_24h = np.zeros(n, dtype=np.float64)
    uid_product_novelty = np.zeros(n, dtype=np.int64)
    new_device_for_uid = np.zeros(n, dtype=np.int64)
    dist1_to_uid_median_ratio = np.zeros(n, dtype=np.float64)

    uid_ts = {}
    uid_amts = {}
    uid_prods = {}
    uid_devs = {}
    uid_dist1 = {}

    for i in range(n):
        row = df.iloc[i]
        uid = str(row["uid"])
        dt = int(row["TransactionDT"])
        amt = float(row["TransactionAmt"])
        product = str(row.get("ProductCD", ""))

        pts = uid_ts.get(uid, [])
        pamts = uid_amts.get(uid, [])
        pprods = uid_prods.get(uid, set())
        pdevs = uid_devs.get(uid, set())
        pdist = uid_dist1.get(uid, [])

        uid_prior_frequency[i] = len(pts)

        if len(pamts) > 0:
            amt_to_uid_median_ratio[i] = min(amt / max(float(np.median(pamts)), EPSILON), 5.0)

        uid_txn_count_24h[i] = sum(1 for t in pts if dt - t <= SECONDS_24H)
        uid_amt_sum_24h[i] = sum(a for t, a in zip(pts, pamts) if dt - t <= SECONDS_24H)

        uid_product_novelty[i] = 0 if product in pprods else 1

        dev_sig = device_signature(row)
        if dev_sig is not None:
            new_device_for_uid[i] = 0 if dev_sig in pdevs else 1

        dist1_val = row.get("dist1")
        if pd.notna(dist1_val) and len(pdist) > 0:
            dist1_to_uid_median_ratio[i] = min(float(dist1_val) / max(float(np.median(pdist)), EPSILON), 5.0)

        # Update AFTER (causal)
        uid_ts.setdefault(uid, []).append(dt)
        uid_amts.setdefault(uid, []).append(amt)
        uid_prods.setdefault(uid, set()).add(product)
        if dev_sig is not None:
            uid_devs.setdefault(uid, set()).add(dev_sig)
        if pd.notna(dist1_val):
            uid_dist1.setdefault(uid, []).append(float(dist1_val))

        if i % 100000 == 0 and i > 0:
            print(f"  Feature progress: {i}/{n} rows ({i*100//n}%)")

    df["uid_prior_frequency"] = uid_prior_frequency
    df["amt_to_uid_median_ratio"] = amt_to_uid_median_ratio
    df["uid_txn_count_24h"] = uid_txn_count_24h
    df["uid_amt_sum_24h"] = uid_amt_sum_24h
    df["uid_product_novelty"] = uid_product_novelty
    df["new_device_for_uid"] = new_device_for_uid
    df["dist1_to_uid_median_ratio"] = dist1_to_uid_median_ratio

    for col in raw_c_features + raw_d_features + raw_v_features:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)

    raw_m_default_codes = {}
    for col in raw_m_features:
        cat_series = df[col].astype("string").fillna("missing")
        categories = sorted(cat_series.dropna().unique().tolist())
        if "missing" not in categories:
            categories.append("missing")
            categories = sorted(categories)
        encoded = pd.Categorical(cat_series, categories=categories).codes.astype(np.int32)
        df[col] = encoded
        raw_m_default_codes[col] = int(categories.index("missing"))

    output_cols = (
        IEEE_CIS_TRAINING_FEATURES
        + raw_c_features
        + raw_d_features
        + raw_m_features
        + raw_v_features
        + ["isFraud"]
    )
    return (
        df[output_cols].copy(),
        raw_c_features,
        raw_d_features,
        raw_m_features,
        raw_v_features,
        raw_m_default_codes,
    )

print("Building features (this may take several minutes on full data)...")
features_df, RAW_C_FEATURES, RAW_D_FEATURES, RAW_M_FEATURES, RAW_V_FEATURES, RAW_M_DEFAULT_CODES = build_ieee_features(merged)
print(f"Done! Shape: {features_df.shape}")
print(
    f"Raw masked features included: C={len(RAW_C_FEATURES)}, D={len(RAW_D_FEATURES)}, "
    f"M={len(RAW_M_FEATURES)}, V={len(RAW_V_FEATURES)}"
)
features_df.describe()


## 5. Train / Validation Split

In [ ]:
feature_cols = IEEE_CIS_TRAINING_FEATURES + RAW_C_FEATURES + RAW_D_FEATURES + RAW_M_FEATURES + RAW_V_FEATURES
TARGET = "isFraud"

split_idx = int(len(features_df) * TRAIN_RATIO)
train_df = features_df.iloc[:split_idx]
val_df = features_df.iloc[split_idx:]

X_train = train_df[feature_cols].values
y_train = train_df[TARGET].values.astype(int)
X_val = val_df[feature_cols].values
y_val = val_df[TARGET].values.astype(int)

n_neg = int(np.sum(y_train == 0))
n_pos = max(int(np.sum(y_train == 1)), 1)
scale_pos_weight = n_neg / n_pos

print(f"Train: {len(train_df)} rows ({np.mean(y_train):.4f} fraud rate)")
print(f"Val:   {len(val_df)} rows ({np.mean(y_val):.4f} fraud rate)")
print(f"scale_pos_weight: {scale_pos_weight:.1f}")
print(f"Training feature count: {len(feature_cols)}")

## 6. Train XGBoost

In [ ]:
import xgboost as xgb

# Prefer GPU on Kaggle when available; otherwise fall back to CPU
try:
    probe_X = np.array([[0.0, 0.0], [1.0, 1.0]])
    probe_y = np.array([0, 1])
    _t = xgb.XGBClassifier(
        device="cuda",
        n_estimators=1,
        max_depth=1,
        tree_method="hist",
        eval_metric="aucpr",
        random_state=42,
    )
    _t.fit(probe_X, probe_y, verbose=False)
    device = "cuda"
except Exception as exc:
    device = "cpu"
    print(f"GPU unavailable, falling back to CPU: {exc}")
print(f"Using device: {device}")

model = xgb.XGBClassifier(
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
    learning_rate=LEARNING_RATE,
    scale_pos_weight=scale_pos_weight,
    device=device,
    tree_method="hist",
    eval_metric="aucpr",
    use_label_encoder=False,
    random_state=42,
)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=10)

## 7. Evaluate

In [ ]:
from sklearn.metrics import (
    average_precision_score, confusion_matrix,
    f1_score, precision_score, recall_score,
    precision_recall_curve,
)

y_pred_proba = model.predict_proba(X_val)[:, 1]
y_pred = (y_pred_proba >= 0.5).astype(int)

pr_auc = average_precision_score(y_val, y_pred_proba)
prec = precision_score(y_val, y_pred, zero_division=0)
rec = recall_score(y_val, y_pred, zero_division=0)
f1 = f1_score(y_val, y_pred, zero_division=0)
cm = confusion_matrix(y_val, y_pred)

metrics = {
    "threshold": 0.5,
    "precision": round(float(prec), 4),
    "recall": round(float(rec), 4),
    "f1": round(float(f1), 4),
    "pr_auc": round(float(pr_auc), 4),
    "confusion_matrix": cm.tolist(),
}

print("=" * 50)
print("  Classification Report")
print("=" * 50)
print(f"  Precision:  {metrics['precision']}")
print(f"  Recall:     {metrics['recall']}")
print(f"  F1 Score:   {metrics['f1']}")
print(f"  PR-AUC:     {metrics['pr_auc']}")
print(f"\n  Confusion Matrix:")
print(f"    TN={cm[0][0]:>6}  FP={cm[0][1]:>6}")
print(f"    FN={cm[1][0]:>6}  TP={cm[1][1]:>6}")
print("=" * 50)

## 8. Suggest Thresholds

In [ ]:
# Keep thresholds aligned with Phase 4A serving semantics.
thresholds = {
    "level": {
        "low_to_medium": 0.4,
        "medium_to_high": 0.75,
    },
    "decision": {
        "allow_to_review": 0.4,
        "review_to_block": 0.9,
    },
}

print(f"Level thresholds:    {thresholds['level']}")
print(f"Decision thresholds: {thresholds['decision']}")


## 9. Export Artifacts

In [ ]:
import json
import joblib
import shutil
from pathlib import Path
from datetime import datetime, timezone

UID_RAW_FIELDS = ["card1", "card2", "card3", "card5", "addr1", "addr2"]
ENGINEERED_REQUIRED_RAW = [
    "TransactionDT", "TransactionAmt", "ProductCD",
    *UID_RAW_FIELDS,
    "P_emaildomain", "R_emaildomain",
    "DeviceType", "DeviceInfo", "dist1",
    "id_30", "id_31", "id_33",
]
REQUIRED_RAW = list(dict.fromkeys(
    ENGINEERED_REQUIRED_RAW + RAW_C_FEATURES + RAW_D_FEATURES + RAW_M_FEATURES + RAW_V_FEATURES
))

manifest = {
    "domain": "ieee_cis",
    "artifact_version": ARTIFACT_VERSION,
    "model_family": "xgboost",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "feature_order": feature_cols,
    "required_raw_fields": REQUIRED_RAW,
    "online_features": IEEE_CIS_ONLINE_FEATURES,
    "training_features": feature_cols,
    "score_mapping": {
        "heuristic_score": "scores.heuristic",
        "supervised_probability": "scores.supervised",
        "final_risk_score": "risk.score",
    },
    "alert_thresholds": {
        "low_to_medium": thresholds["level"]["low_to_medium"],
        "medium_to_high": thresholds["level"]["medium_to_high"],
        "allow_to_review": thresholds["decision"]["allow_to_review"],
        "review_to_block": thresholds["decision"]["review_to_block"],
    },
}

metadata = {
    "exported_at": datetime.now(timezone.utc).isoformat(),
    "txn_data_source": TXN_PATH,
    "identity_data_source": IDENTITY_PATH,
    "sample_rows": SAMPLE_ROWS,
    "train_rows": len(train_df),
    "val_rows": len(val_df),
    "device": device,
    "n_estimators": N_ESTIMATORS,
    "max_depth": MAX_DEPTH,
    "learning_rate": LEARNING_RATE,
    "scale_pos_weight": round(scale_pos_weight, 2),
    "training_feature_count": len(feature_cols),
    "online_feature_count": len(IEEE_CIS_ONLINE_FEATURES),
    "required_raw_field_count": len(REQUIRED_RAW),
}

feature_defaults = {f: 0.0 for f in feature_cols}
for col, code in RAW_M_DEFAULT_CODES.items():
    feature_defaults[col] = float(code)

version_dir = Path(OUTPUT_DIR) / "ieee_cis" / ARTIFACT_VERSION
version_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(model, version_dir / "supervised_model.joblib")

for name, data in [
    ("manifest.json", manifest),
    ("feature_order.json", feature_cols),
    ("feature_defaults.json", feature_defaults),
    ("thresholds.json", thresholds),
    ("metrics.json", metrics),
    ("metadata.json", metadata),
]:
    (version_dir / name).write_text(json.dumps(data, indent=2, default=str) + "\n")

manifests_dir = Path(OUTPUT_DIR) / "manifests"
manifests_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(version_dir / "manifest.json", manifests_dir / "ieee_cis_latest.json")

print(f"\n✅ Artifacts exported to: {version_dir}")
print(f"✅ Latest manifest: {manifests_dir / 'ieee_cis_latest.json'}")
print(f"\n📦 Download the 'artifacts/' folder from the Output tab.")


## 10. Verify Output

In [ ]:
import os

print("Exported files:")
for root, dirs, files in os.walk(OUTPUT_DIR):
    level = root.replace(OUTPUT_DIR, "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = " " * 2 * (level + 1)
    for file in sorted(files):
        size_kb = os.path.getsize(os.path.join(root, file)) / 1024
        print(f"{subindent}{file} ({size_kb:.1f} KB)")